<a href="https://colab.research.google.com/github/Kelmoir/CAFA6_experimenting/blob/main/CAFA6_protBert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model building fr the CAFA 6 Challenge, with transformers

This is just model building forthe CAFA6 Challenge on Kaggle: https://www.kaggle.com/competitions/cafa-6-protein-function-prediction/overview

In this notebook, I will use Huggingface transformers with the pretrained ProtBERT Model: https://huggingface.co/Rostlab/prot_bert

Basic exploration was done externally, thus won't be repeated in here.

The gist:
- GO factors are derived from proteins
- The list of them is usually not complete
- GO-terms are usually part of a graph like structure, a single one can't exist alone. See: https://geneontology.org/docs/ontology-documentation/
- 82k proteis, 426k of possible GO-Terms
- This is a multi-label classification, thus The GO-terms will need to one-hot-encoded
- ProtBert model should provide base Protein understanding, more data augmeentation will need to happen on the GO-term side.

Provided data:
train_sequences.fasta – amino acid sequences for proteins in the training set
train_terms.tsv – the training set of proteins and corresponding annotated GO terms
train_taxonomy.tsv – taxon IDs for proteins in the training set
go-basic.obo – ontology graph structure
testsuperset.fasta – amino acid sequences for proteins on which predictions should be made
testsuperset-taxon-list.tsv – taxon IDs for proteins in the test superset
IA.tsv – information accretion for each term (used to weight precision and recall)
sample_submission.tsv – sample submission file in the correct format

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Import the librays
try:
  import datasets
  import gradio as gr
  import torchmetrics
  import pycocotools
  import evaluate
  import pandas as pd
  import obonet
  import numpy as np
  from Bio import SeqIO
except ModuleNotFoundError:
  !pip install datasets gradio
  !pip install torchmetrics[detection]
  !pip install biopython
  !pip install obonet
  !pip install evaluate

  import evaluate
  import datasets
  import gradio as gr

  # Required for evaluation
  import torchmetrics
  import pycocotools # make sure we got this for torchmetrics

  import pandas as pd
  import obonet
  import numpy as np
  from Bio import SeqIO

import random

import numpy as np

import torch
import transformers

from typing import List
from datasets import Value

MAX_LENGTH = 1024

print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")
print(f"Using torchmetrics version: {torchmetrics.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
Using transformers version: 5.0.0
Using datasets version: 4.0.0
Using torch version: 2.10.0+cu128
Using torchmetrics version: 1.8.2


In [ ]:
from pathlib import Path
my_path = Path('/content/drive/MyDrive/CAFA6')
my_path.mkdir(exist_ok=True)
dataset_dir = my_path / "data/dataset"
model_data_path=my_path / "data/model"
model_data_path.mkdir(exist_ok=True)
demo_path = Path("../demos")
demo_path.mkdir(exist_ok=True)

In [ ]:
import os
def clear_model_folder():
  for item in model_data_path.iterdir():
    # print(item)
    os.remove(item)

#clear_model_folder()

## Data preparation

The data is acquired, and places in "drive/CAFA6/data/dataset" (dataset_dir)

We need to:
- generate a dict of Protein names, and their sequences (upper case, for the tokenizer, see: https://huggingface.co/Rostlab/prot_bert)
- turn the GO-terms into a one-hot encoded dataFrame, with an additional column for the Prozein mae or sequence... -> this is part of the model preprocess tpieline (with_transform...)
- account for validation data

### Full data generation from source files

In [ ]:
prot_sequences ={}
unique_prot = []
for sequence in SeqIO.parse(dataset_dir/"Train/train_sequences.fasta", "fasta"):
  unique_prot.append(sequence.id.split("|")[1])
  prot_sequences[sequence.id.split("|")[1]] = str(sequence.seq).upper().replace(r"[UZOB]", "X") # Reading the relevant data, and pre-formatting it for ProtBERT
train_df = pd.read_csv(dataset_dir/"Train/train_terms.tsv", delimiter="\t")
obo_terms = obonet.read_obo(dataset_dir/"Train/go-basic.obo")
unique_go_terms = np.unique(train_df["term"])
# generate id2label and label2id datasets for the model to use
id2label = {id: label for id, label in enumerate(unique_go_terms)}
label2id = {label: id for id, label in enumerate(unique_go_terms)}

In [ ]:
# Create a function to view a random sample
def print_random_item():
  rand_int= random.randint(0, len(unique_prot))
  key= unique_prot[rand_int]
  print(f"The Protein ID is: {key}")
  print(f" the sequence for it is: {prot_sequences[key]}")
  print(f" Available GO-terms:\n {train_df['term'][train_df['EntryID'] == key]}")

In [ ]:
print_random_item()

The Protein ID is: Q9UKN5
 the sequence for it is: MHHRMNEMNLSPVGMEQLTSSSVSNALPVSGSHLGLAASPTHSAIPAPGLPVAIPNLGPSLSSLPSALSLMLPMGIGDRGVMCGLPERNYTLPPPPYPHLESSYFRTILPGILSYLADRPPPQYIHPNSINVDGNTALSITNNPSALDPYQSNGNVGLEPGIVSIDSRSVNTHGAQSLHPSDGHEVALDTAITMENVSRVTSPISTDGMAEELTMDGVAGEHSQIPNGSRSHEPLSVDSVSNNLAADAVGHGGVIPMHGNGLELPVVMETDHIASRVNGMSDSALSDSIHTVAMSTNSVSVALSTSHNLASLESVSLHEVGLSLEPVAVSSITQEVAMGTGHVDVSSDSLSFVSPSLQMEDSNSNKENMATLFTIWCTLCDRAYPSDCPEHGPVTFVPDTPIESRARLSLPKQLVLRQSIVGAEVGVWTGETIPVRTCFGPLIGQQSHSMEVAEWTDKAVNHIWKIYHNGVLEFCIITTDENECNWMMFVRKARNREEQNLVAYPHDGKIFFCTSQDIPPENELLFYYSRDYAQQIGVPEHPDVHLCNCGKECNSYTEFKAHLTSHIHNHLPTQGHSGSHGPSHSKERKWKCSMCPQAFISPSKLHVHFMGHMGMKPHKCDFCSKAFSDPSNLRTHLKIHTGQKNYRCTLCDKSFTQKAHLESHMVIHTGEKNLKCDYCDKLFMRRQDLKQHVLIHTQERQIKCPKCDKLFLRTNHLKKHLNSHEGKRDYVCEKCTKAYLTKYHLTRHLKTCKGPTSSSSAPEEEEEDDSEEEDLADSVGTEDCRINSAVYSADESLSAHK
 Available GO-terms:
 481069    GO:1990837
481070    GO:0005515
Name: term, dtype: object


### Load the prepared dataset

In [ ]:
# generate the unique_go_terms, label2id and id2label with minimal memory overhead

train_df = pd.read_csv(dataset_dir/"Train/train_terms.tsv", delimiter="\t")
# Count the oboterms provided by the obo file
unique_go_terms = np.unique(train_df["term"])
  # generate id2label and label2id datasets for the model to use
id2label = {id: label for id, label in enumerate(unique_go_terms)}
label2id = {label: id for id, label in enumerate(unique_go_terms)}
del train_df

### Loading Pre-trained model and tokenizer

In [ ]:
from google.colab import userdata
# Generate required items from the model
from transformers import AutoModelForSequenceClassification, BertTokenizerFast
def create_model():
  return AutoModelForSequenceClassification.from_pretrained("Rostlab/prot_bert",
                                    num_labels= len(unique_go_terms),
                                    id2label = id2label,
                                    label2id=label2id,
                                    token=userdata.get("Huggingface"))
def create_tokenizer():
  return BertTokenizerFast.from_pretrained("Rostlab/prot_bert",
                                       do_lower_case=False,
                                       token=userdata.get("Huggingface"))

model = create_model()
tokenizer = create_tokenizer()

config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

### Number of used GO-Terms

Even though the training data only contains 26125 GO-Terms, those are less tan the 40122 GO-terms known to the obonet file.

This may become relevant, if ancestor GO-terms or child GO-terms are not part of the training set.

Preditions on non-visible GO-terms might be relevant on the testing set, and data enhancement might yield the GO-terms, tat are not yet in the training sequence - the obonet file GO-terms will be used.

Why is that difference there? are some GO-Terms not relevant for humans?

For that, look at the ia.tsv - it also got 40122 entrys, but about 18077 have a weight of 0. 40122-18077 = 22045, which is less than the GO-terms in the train.tsv.

This leads to the assumption, that the missing GO-terms are simply not part of the training set/are considered irrelevant for the purpose of the competition.

### Generating the dataset

We we will use a default huggingface dataset structure to host the data. Since this is powered by apache arrow, it is made for these aplications. We  will form it from a dictionary, and store it, afterwards.

We do minimal proccessing (uppcase, UZOB replacement) processing, butr we can repeat those for the model internal preprocessing, to have a dataset, that is as clean as possible

In [ ]:
# Generate the dataset. Apparently, we don't need to worry about memory too much, if we post-pone the preprocessing (mainly the one-hot encoding, but also the prot_seq padding)
from tqdm.auto import tqdm as progressbar
def generate_dataset(unique_prot = unique_prot, train_df = train_df, max_length=1024):
  """
  Generates a transformers.Dataset from the available data
  It will also remove items, where the protein sequence is too long, because then the model will require too much RAM when training.
  """
  # we got unique_prot already
  sequence_list = []
  go_term_list = [] # list of " " separated GO-terms
  prot_list = []
  last_prot = train_df["EntryID"][0]
  go_term_string = ""
  for item in progressbar(train_df.itertuples(), total=train_df.shape[0], desc="creating data as alligned lists"):
    if item[1] != last_prot:
      if len(prot_sequences[last_prot]) <= max_length:
        go_term_list.append(go_term_string) # Processing the go-term string, when we throw the protein away, because it is too big, is rather pointless.
        sequence_list.append(prot_sequences[last_prot])
        prot_list.append(last_prot)
      last_prot = item[1]
      go_term_string = item[2]
    else:
      go_term_string = go_term_string + " " + item[2]
  #finish last item
  if len(prot_sequences[last_prot]) <= max_length:
    go_term_list.append(go_term_string)
    sequence_list.append(prot_sequences[last_prot])
    prot_list.append(last_prot)
  data_dict = {"protein": prot_list, "sequence": sequence_list, "go_term_list": go_term_list}
  print(f"[INFO Number of proteins in the dataset: {len(prot_list)}")
  print(f"[INFO] Number of proteins removed from data source: {len(prot_sequences) - len(prot_list)}")
  return datasets.Dataset.from_dict(data_dict)

In [ ]:
for item in train_df.head(10).itertuples():
  print(item)
train_df.head()

Pandas(Index=0, EntryID='Q5W0B1', term='GO:0000785', aspect='C')
Pandas(Index=1, EntryID='Q5W0B1', term='GO:0004842', aspect='F')
Pandas(Index=2, EntryID='Q5W0B1', term='GO:0051865', aspect='P')
Pandas(Index=3, EntryID='Q5W0B1', term='GO:0006275', aspect='P')
Pandas(Index=4, EntryID='Q5W0B1', term='GO:0006513', aspect='P')
Pandas(Index=5, EntryID='Q5W0B1', term='GO:0003682', aspect='F')
Pandas(Index=6, EntryID='Q5W0B1', term='GO:0005515', aspect='F')
Pandas(Index=7, EntryID='Q3EC77', term='GO:0000138', aspect='C')
Pandas(Index=8, EntryID='Q3EC77', term='GO:0005794', aspect='C')
Pandas(Index=9, EntryID='Q8IZR5', term='GO:0005515', aspect='F')


,EntryID,term,aspect
0,Q5W0B1,GO:0000785,C
1,Q5W0B1,GO:0004842,F
2,Q5W0B1,GO:0051865,P
3,Q5W0B1,GO:0006275,P
4,Q5W0B1,GO:0006513,P


In [ ]:
# Generate the actual dataset and save it to Google drive
dataset_save_path = model_data_path/f"full_dataset_max{MAX_LENGTH}"
generate_dataset(max_length = MAX_LENGTH).save_to_disk(dataset_path=dataset_save_path)
print(f"[INFO] Stored the dataset in {dataset_save_path}, with a maximum protein_length of {MAX_LENGTH}")

creating data as alligned lists:   0%|          | 0/537027 [00:00<?, ?it/s]

[INFO Number of proteins in the dataset: 78481
[INFO] Number of proteins removed from data source: 3923


Saving the dataset (0/1 shards):   0%|          | 0/78481 [00:00<?, ? examples/s]

[INFO] Stored the dataset in /content/drive/MyDrive/CAFA6/data/model/full_dataset_max1024, with a maximum protein_length of 1024


In [ ]:
test_dataset_2 = generate_dataset(train_df = train_df.head(1000))
test_dataset_2["protein"][0], test_dataset_2["go_term_list"][0],train_df["term"][train_df["EntryID"] == test_dataset_2["protein"][0]]

creating data as alligned lists:   0%|          | 0/1000 [00:00<?, ?it/s]

Number of proteins: 189


('Q5W0B1',
 ' GO:0000785 GO:0004842 GO:0051865 GO:0006275 GO:0006513 GO:0003682 GO:0005515',
 0    GO:0000785
 1    GO:0004842
 2    GO:0051865
 3    GO:0006275
 4    GO:0006513
 5    GO:0003682
 6    GO:0005515
 Name: term, dtype: object)

### Loading the prepared datset

it was stored in `model_data_path/"full_dataset"`

We will also immediately change it to the train test splits

In [ ]:
# Create train, test, validation split
# Split the data into train and test splits
def create_train_val_split_from_dataset(dataset):
  dataset_split = dataset.train_test_split(test_size=0.3, # 30% of dsata will be the test split
                                                    seed=42)
  dataset_test_val_split = dataset_split["test"].train_test_split(test_size=0.66, # This means, 66% of the data will be the new test split, and the remainder the validataion data
                                        seed=42)
  # result_dataset["train"] = dataset_split["train"]
  # result_dataset["validation"] = dataset_test_val_split["train"]
  # result_dataset["test"] = dataset_test_val_split["test"]

  return datasets.DatasetDict({
      "train": dataset_split["train"],
      "validation": dataset_test_val_split["train"],
      "test": dataset_test_val_split["test"]
  })

In [ ]:
dataset = create_train_val_split_from_dataset(datasets.load_from_disk(dataset_path=model_data_path/"full_dataset_max1024"))

#dataset = create_train_val_split_from_dataset(datasets.load_from_disk(dataset_path=dataset_save_path))

## Helper functions and general setup

### Create a preprocessing function

The dataset has 2 main points left to be processed:
The sequences won't dit the tokenizer (spaces are neded after each Letter), and the GO-terms need to be propperly endoced. Since we don't got location data, we go for the usual one-hot-encoding

Thus we ned to:
1. Load the data in batch form
2. Process all the sequence strings
3. Tokenize them
4. Pass them back to the data
5. Take in the list of GO-terms (separated by " ")
6. One-hot encode them
7. feed that back into the data
8. Return everything

2, 3, 4 and 5, 6, 7 can be done at the same time, though.

In [ ]:
def preprocess_batch(examples, tokenizer = tokenizer, transforms = None, max_length=1024):
  """
  Preprocesses a batch of items from the dataset, and gets them ready for model training.
  Removes padding from tokenizer calls, as padding will be handled by the data_collate_function.

  Parameters:
  **examples** a dataset items (dict of lists), which contain the preprocessed samples.
  **tokenizer** an instance of the ProtBERT tokenizer, in order to perform the tokenization.
  **transforms** Any data-augmeentation transform instructions. None available, atm.
  **max_length** The maximum sequence length for tokenization. Sequences longer than this will be truncated.

  Returns: preprocessed dataset items (dict of lists), ready for passing to the model
  """
  # Common part for both batched and single example processing
  sequences_processed = [prepare_for_tokenization(item) for item in examples["sequence"]] if isinstance(examples["sequence"], list) else [prepare_for_tokenization(examples["sequence"])]

  # Tokenize with truncation, return as Python lists of integers
  # `truncation=True` added to handle sequences longer than max_length
  tokenized_output = tokenizer(sequences_processed, max_length=max_length, truncation=True)

  # Process go-terms - if they are there. The final test dataset will not have them
  labels_list = []
  if "go_term_list" in examples:
    # Common part for both batched and single example processing
    go_term_lists_processed = examples["go_term_list"] if isinstance(examples["go_term_list"], list) else [examples["go_term_list"]]

    # Prepare labels as 1D tensors
    for go_terms_str in go_term_lists_processed:
      labels = torch.full((len(unique_go_terms),), 0, dtype=torch.float)
      for item in go_terms_str.split(" "):
        if item: # Ensure item is not an empty string
            labels[label2id[item]] = 1
      labels_list.append(labels)

  # When batched=True, 'examples' is a dict of lists. We need to return a dict of lists of individual tensors.
  # When not batched (e.g., accessing a single element), 'examples' is a dict of single items. We need to return a dict of single tensors.

  if isinstance(examples["protein"], list): # This is the 'batched=True' path for datasets.with_transform
    output_dict = {
        "protein": examples["protein"],
        "input_ids": [torch.tensor(ids, dtype=torch.long) for ids in tokenized_output["input_ids"]],
        "token_type_ids": [torch.tensor(ids, dtype=torch.long) for ids in tokenized_output["token_type_ids"]],
        "attention_mask": [torch.tensor(ids, dtype=torch.long) for ids in tokenized_output["attention_mask"]]
    }
    if "go_term_list" in examples:
        output_dict["labels"] = labels_list # Already list of 1D tensors
    return output_dict
  else: # This is the single item path (e.g. for dataset[0] access)
    output_dict = {
        "protein": examples["protein"],
        "input_ids": torch.tensor(tokenized_output["input_ids"][0], dtype=torch.long),
        "token_type_ids": torch.tensor(tokenized_output["token_type_ids"][0], dtype=torch.long),
        "attention_mask": torch.tensor(tokenized_output["attention_mask"][0], dtype=torch.long)
    }
    if "go_term_list" in examples:
        output_dict["labels"] = labels_list[0] # Single 1D tensor
    return output_dict

### Prepare for tokenization

In [ ]:
# Generate an additional input adjuster for the tokenizer
def prepare_for_tokenization(input_prot_seq: str)-> str:
  """
  This function shall insert a [SPACE] between each protein sequence item, and returns the resulting string,
  """
  input_prot_seq = input_prot_seq.strip()
  spaces = " "*len(input_prot_seq)

# Source - https://stackoverflow.com/a
# Posted by Ma0, modified by community. See post 'Timeline' for change history
# Retrieved 2026-01-12, License - CC BY-SA 3.0
  return ''.join(map(''.join, zip(input_prot_seq, spaces))).strip()  # kudos @Coldspeed

### Cuda config and finish the preprocess setup and application

In [ ]:
# Set PyTorch CUDA allocation configuration for improved memory management
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from functools import partial



preprocess_batch_partial = partial(preprocess_batch,
                                   tokenizer = tokenizer,
                                   transforms = None,
                                   max_length = MAX_LENGTH)

preprocess_batch_partial;

In [ ]:
# Apply our partial function to our datasets
processed_dataset = dataset.copy();

# Apply processing function
processed_dataset["train"] = processed_dataset["train"].with_transform(transform=preprocess_batch_partial)
processed_dataset["validation"] = processed_dataset["validation"].with_transform(transform=preprocess_batch_partial)
processed_dataset["test"] = processed_dataset["test"].with_transform(transform=preprocess_batch_partial)

### Create a data collation function

The copllation function is supposed to gather the data into batches in order to speed up the process, by filling the GPU memory nicely.

In [ ]:
from typing import List, Dict, Any
import torch

def data_collate_function(preprocessed_batch: List[Dict[str, Any]])-> Dict[str, Any]:
  """
  Stacks together groups of preprocessed samples into batches for our model,
  applying dynamic padding to the maximum sequence length within the batch.

  Params:
  **preprocessed_batch**: A list of Dicts, where each Dict represents a single processed sample
                          with unpadded 1D tensors for input_ids, token_type_ids, attention_mask, and labels.
  """
  collated_data = {}

  # Determine the maximum sequence length in the current batch
  max_seq_len = max([len(sample["input_ids"]) for sample in preprocessed_batch])

  # Manually pad each item in the batch
  padded_input_ids = []
  padded_token_type_ids = []
  padded_attention_mask = []
  labels = []

  for sample in preprocessed_batch:
    current_len = len(sample["input_ids"])
    padding_len = max_seq_len - current_len

    # Pad input_ids with the tokenizer's pad_token_id
    padded_input_ids.append(torch.cat([sample["input_ids"], torch.full((padding_len,), tokenizer.pad_token_id, dtype=torch.long)]))
    # Pad token_type_ids with 0
    padded_token_type_ids.append(torch.cat([sample["token_type_ids"], torch.full((padding_len,), 0, dtype=torch.long)]))
    # Pad attention_mask with 0
    padded_attention_mask.append(torch.cat([sample["attention_mask"], torch.full((padding_len,), 0, dtype=torch.long)]))
    if "labels" in sample:
      labels.append(sample["labels"])

  # Stack the padded tensors to form the batch
  collated_data["input_ids"] = torch.stack(padded_input_ids)
  collated_data["token_type_ids"] = torch.stack(padded_token_type_ids)
  collated_data["attention_mask"] = torch.stack(padded_attention_mask)
  if labels != []:
    collated_data["labels"] = torch.stack(labels)

  return collated_data

### Function to derive the GO-Terms from the logits

Needs to be able to take in a list of items, or just a single Items, and shall return a list of applicable GO-terms in the compretition prediction format. We will need a threshold here.

Sample submission Snippet:
```
P9WHI7   GO:0009274   0.931   
P9WHI7   GO:0071944   0.540
P9WHI7   GO:0005575   0.324
P04637   GO:1990837   0.23
P04637   GO:0031625   0.989
P04637   GO:0043565   0.64
P04637   GO:0001091   0.49
```

Note: No header, .tsv and to include up to 3 digits of the score

In [ ]:
def process_results(proteins, logits, threshold=0.6, top_n=10):
  """
  This function will take in a sample of protein id(s) and map that to a list of Logit outputs, mapped to the proteins.
  Then, this function will return a headerless pd.Dataframe, containing the results. (protein ID, Go-tern, score)
  Args:
  **proteins** A list of protein ids, or a single one
  **logits** A list of Logit (torch.float) or a single Logit array
  **threshold** A float value between 0 and 1. Only GO-terms with a score higher than this value will be considered.
  **top_n** An integer value. If no GO-terms are above the threshold, the top_n GO-terms will be returned.
  Ensure, that the length is equal
  """
  # Convert the single item case into a list/tensor, to enable iterating over them
  if isinstance(proteins, str):
    proteins = [proteins]
    logits = [logits] # Ensure logits is also a list for iteration

  all_results = []
  for i, protein_id in enumerate(proteins):
    protein_logits = logits[i]
    # Convert logits to probabilities using sigmoid
    probabilities = torch.sigmoid(protein_logits).squeeze()

    # Get indices of GO terms above threshold
    above_threshold_indices = torch.where(probabilities > threshold)[0]

    # If no terms above threshold, take the top_n terms
    if len(above_threshold_indices) == 0:
      top_n_values, top_n_indices = torch.topk(probabilities, k=min(top_n, len(probabilities)))
      selected_indices = top_n_indices
      selected_probabilities = top_n_values
    else:
      selected_indices = above_threshold_indices
      selected_probabilities = probabilities[above_threshold_indices]

    # Sort selected terms by probability in descending order
    sorted_probabilities, sort_indices = torch.sort(selected_probabilities, descending=True)
    sorted_indices = selected_indices[sort_indices]

    for idx, prob in zip(sorted_indices, sorted_probabilities):
      go_term = id2label[idx.item()]
      score = prob.item()
      all_results.append([protein_id, go_term, f"{score:.3f}"]) # Format score to 3 decimal places

  results_df = pd.DataFrame(all_results)
  # The request specifies 3 columns, all string, no headers, no index
  return results_df

### Evaluation function

The default evaluations are generally amost always close to perfect scores - Because 26K Go terms, of which 99.9% are 0, and predicted as 0. Thus I added the active loss to reduce the evaluated sample to the GO-terms, that are 1 or predicted 1 in order to get a more suitable loss function.

In [ ]:
def compute_active_loss(predictions: np.array, references: np.array) -> float:
  """
  Calculates the loss of active labels or active predicted items.
  Essentially, when neither the labels or predictions are true, the item is ignored.

  As in:
  |Predicted | labels | Result     |
  |----------|--------|------------|
  |   0      |   0    | Ignored    |
  |   0      |   1    | add to loss|
  |   1      |   0    | add to loss|
  |   1      |   1    | add to OK  |

  This is done because usually <0.1% of the labels and predictions are actually true, leading to ludicrously
  high and hard to understand scores.
  """
  # Find elements where either prediction or reference is 1
  active_elements_mask = (predictions == 1) | (references == 1)

  # Filter predictions and references to only include active elements
  active_predictions = predictions[active_elements_mask]
  active_references = references[active_elements_mask]
  #print(f"[INFO] Number of active predicitons = {len(active_predictions)}")
  #print(f"[INFO] Number of active references = {len(active_references)}")

  if len(active_predictions) and len(active_references) == 0:
    return 0.0 # No active elements, so no active loss

  # Calculate the absolute difference for active elements.
  # A difference of 1 means a mismatch (0 vs 1 or 1 vs 0), which contributes to loss.
  # A difference of 0 means a match (1 vs 1), which does not contribute to loss here.
  loss_elements = np.ones(len(active_predictions))[active_predictions != active_references]
  #print(f"[INFO] Number of wrong items = {np.sum(loss_elements)}")
  #loss_per_active_element = np.abs(active_predictions - active_references)

  # Sum these differences to get the total active loss
  total_active_loss = np.sum(loss_elements) / len(active_predictions)

  return float(total_active_loss)

In [ ]:
import evaluate
import numpy as np
import torch
from typing import Tuple

THRESHOLD = 0.6

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def do_evaluation(predictions_and_labels: Tuple[np.array, np.array], threshold = THRESHOLD):
  """
  Computes the accuracy, precision, and recall for multi-label classification.
  Expected input: predictions (logits), labels (one-hot encoded).
  """
  predictions, labels = predictions_and_labels
  accuracy_list = []
  precision_list = []
  recall_list = []
  active_loss_list = []

  # We can either get direcly aray, or a list contraining the arrays. List length is number of bartches, array shape is (num_samples, num_labels)
  if not isinstance(predictions_and_labels, transformers.trainer_utils.EvalPrediction):
    if len(predictions.shape) == 1:
      predictions = [np.array([predictions])]
      labels = [np.array([labels])]
    elif len(predictions.shape) == 2:
      predictions = [predictions]
      labels = [labels]
    else:
      raise ValueError(f"Predictions shall be a 1.-dim numpoy array, a 2-dim numpy array or transformers.trainer_utils.EvalPrediction. But thes are of shape: {predictions.shape}")

  for pred_item, label_item in zip(predictions, labels):
    # Convert raw predictions (logits) to probabilities
    probabilities = torch.sigmoid(torch.tensor(pred_item))

    # Binarize probabilities using the threshold and convert to int32
    binary_predictions = (probabilities >= threshold).int().numpy()

    # Ensure labels are also int32
    labels_np = label_item.astype(np.int32)

    # print(f"Shape: {labels_np.shape}")

    # For multi-label classification, it's common to use 'micro', 'macro', or 'samples' averaging.
    # 'micro' is often used in competitions like CAFA for overall performance.
    for pred, label in zip(binary_predictions, labels_np):
      # print(f"Pred: {pred}")
      # print(f"Label: {label}")
      #accuracy_list.append(accuracy_metric.compute(predictions=pred, references=label)["accuracy"])
      #precision_list.append(precision_metric.compute(predictions=pred, references=label, average='micro')["precision"])
      #recall_list.append(recall_metric.compute(predictions=pred, references=label, average='micro')["recall"])
      active_loss_list.append(compute_active_loss(predictions=pred, references=label))

  return {
      #"accuracy": float(np.average(np.array(accuracy_list))),
      #"precision": float(np.average(np.array(precision_list))),
      #"recall": float(np.average(np.array(recall_list))),
      "active_loss": float(np.average(np.array(active_loss_list)))
  }

do_evaluation_partial = partial(do_evaluation,
                                threshold = THRESHOLD)

## Model setup, training

### Counting the parameters on the Model, tryinto vertify, that we set it up, correctly, etc.

- Create function to count parameters
- check the last layer

In [ ]:
tokenizer

BertTokenizer(name_or_path='Rostlab/prot_bert', vocab_size=30, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
def count_params(model):
  """
  Counts the parameters of the model and prints them out
  """
  trainable_params = np.sum([param.numel() for param in model.parameters() if param.requires_grad])
  untrainable_params = np.sum([p.numel() for p in model.parameters() if not p.requires_grad])
  print(f"[INFO] Trainable parameters: {trainable_params}")
  print(f"[INFO] Untrainable parameters: {untrainable_params}")
  print(f"[INFO] Total parameters: {trainable_params + untrainable_params}")
count_params(model)
#with 40122 Go terms: 461056186

[INFO] Trainable parameters: 446709261
[INFO] Untrainable parameters: 0.0
[INFO] Total parameters: 446709261.0


In [ ]:
last_p = None
for p in model.parameters():
  last_p = p
print(len(p))

26125


In [ ]:
# model  # The model has an additional pooling layer before the classification layer - The classification needs to extract 40 times the information from the pooling.

Let's try and pass a batch through the model...

In [ ]:
processed_dataset["train"][:5]

{'protein': ['Q8CHL0', 'O14114', 'Q8N0S2', 'P36646', 'P38164'],
 'labels': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]),
 'input_ids': tensor([[ 2, 21,  6,  ...,  0,  0,  0],
         [ 2, 21, 12,  ...,  5, 18,  3],
         [ 2, 21,  6,  ...,  0,  0,  0],
         [ 2, 21,  6,  ...,  0,  0,  0],
         [ 2, 21,  7,  ...,  0,  0,  0]]),
 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0],
         [0, 0, 0,  ..., 0, 0, 0]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0]])}

In [ ]:
%%time

# Try data_collate_function
example_collated_data_batch = data_collate_function(processed_dataset["train"].select(range(16)))

# Try pass a batch through our model (note: this will be relatively slow if our model is on the CPU)
model = create_model()

with torch.no_grad():
  example_batch_outputs = model(**example_collated_data_batch)
# example_batch_outputs # uncomment for full output
example_batch_outputs.keys()
#with 40122 Go terms: 6 minutes

NameError: name 'create_model' is not defined

In [ ]:
# see the outputs
len(example_batch_outputs["logits"][0]), example_batch_outputs["logits"].dtype

NameError: name 'example_batch_outputs' is not defined

In [ ]:
processed_dataset["train"][:16]["protein"]

['Q8CHL0',
 'O14114',
 'Q8N0S2',
 'P36646',
 'P38164',
 'F4I5N6',
 'P47974',
 'P53593',
 'Q6XJV6',
 'P56756',
 'Q969H8',
 'Q8L7I2',
 'Q16558',
 'Q61847',
 'Q8MYL1',
 'P81919']

In [ ]:
# The protein idds will need to be retrived from the processed dataset
processed_outputs_df = process_results(processed_dataset["train"][:16]["protein"], example_batch_outputs["logits"])
processed_outputs_df.head()

NameError: name 'example_batch_outputs' is not defined

### Creating the model save path
 We will strore our model alongside the training data.


In [ ]:
# We already got model_data_path
model_save_name = "CAFA6_classification_Rostlab-prot_bert"

temp_model_dir = Path("../demos")
temp_model_dir.mkdir(exist_ok = True, parents=True)
model_save_dir = Path(model_data_path, model_save_name)
model_save_dir

PosixPath('/content/drive/MyDrive/CAFA6/data/model/CAFA6_classification_Rostlab-prot_bert')

### Setting up the streaming dataset

Transformers offers streaming datasets, thus, use those as our data is too large for loading into ram. Plus, we got multiple files, to keep filesizes managable - and the memory for loading those.

Documentation: https://huggingface.co/docs/datasets/v4.4.2/en/package_reference/loading_methods#datasets.load_dataset

This turned out to be not neccessary, as the dataset is rather small whithout the one_hot encoding. and the on the fly preprocessing

### Setting up Model training with Training arguments

Intially, I will fllow the general cooing book from mrDBurke: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial#setting-up-a-model-for-training

|Parameters | Value|
|-----------|------|
|`output_dir`| `model_save_dir`|
|`learning_rate`| 0.0001|
|`per_device_train_batch_size`| 32 |
|`per_device_eval_batch_size`| 32 |
|`num_train_epochs`| 10 |
|`eval_strategy` | "epoch" Evaluating per Epoch should prvide accurate results|
|`save_strategy` | "epoch" Dito|
|`save_total_limit` | 1 - The best model should suffice |
|`use_cpu`| "False" |
|`seed` | 42 |
|`load_best_model_at_end` | "True" Why bother about bad models?|
|`logging_strategy` | "epoch" No need for finer logging |
|`report_to` | "none" Local saving shal suffice, for now|
|`push_to_hub`| "False" pushing will be done manually, if there is anything of worth|

In [ ]:
# 3. Set up trainingArguments, see: https://www.learnhuggingface.com/notebooks/hugging_face_object_detection_tutorial#setting-up-our-trainingarguments
from transformers import TrainingArguments
import os
import datetime

BATCH_SIZE =  128 #This is HW dependend, you will usually increae this to max outr the HW memmory
DATA_LOADER_NUM_WORKERS = 4 #os.cpu_count() // 3 # more workers = more faster AND more GPU ram used

# Set the number of epochs to how many lapswe'd like to do on our training data
# Note: typically fine-tuning requires less epochs than training from scratch
NUM_EPOCHS = 6

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0001
MAX_GRAD_NORM = 0.1
WARMUP_STEPS = 50 # warmup for 5% of our training steps

DATE = datetime.date.today()
VERSION_NUMBER = 1

# Create directory and path to save models to
OUTPUT_DIR = Path(demo_path/f"CAFA6_classification_Rostlab_prot_bert_v{VERSION_NUMBER}")
print(f"[INFO] Saving model to: {OUTPUT_DIR}")

# Create TrainingArguments to pass to Trainer
training_args = TrainingArguments(
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    max_grad_norm = MAX_GRAD_NORM,
    warmup_steps = WARMUP_STEPS,
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs = NUM_EPOCHS,  # Optional: train longer
    logging_strategy = "epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    dataloader_num_workers=DATA_LOADER_NUM_WORKERS,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", # evaluate on the eval data, not on the training data
    greater_is_better=False, #for a loss metric, lower is better ... for an eval metric such as MAP, higher is better
    report_to="none", # don't loog metrics to an experiment tracker, like "Weights&Biases"
    push_to_hub=False,
    eval_do_concat_batches=False,
    remove_unused_columns=False,
    gradient_checkpointing=True # Enable gradient checkpointing to save memory
)

[INFO] Saving model to: ../demos/CAFA6_classification_Rostlab_prot_bert_v1


In [ ]:
# Figure out the issues with the do_evaluation
# Create example list of predictions and labels
example_predictions_all_correct = np.array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0])
example_predictions_two_wrong = np.array([0, 1, 0, 0, 5, 0, 0, 0, 0.4, 0])
example_labels = np.array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0])
do_evaluation((example_predictions_one_wrong, example_labels))

NameError: name 'example_predictions_one_wrong' is not defined

In [ ]:
if trainer != None:
  del trainer

### Setup the trainer

In [ ]:
from transformers import Trainer

data_factor = 1
samples_train = int(len(processed_dataset["train"])*data_factor)
samples_validation = int(len(processed_dataset["validation"])*data_factor)
print(f"[INFO] Generating trainer with {samples_train} training items and {samples_validation} validation items.")
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = processed_dataset["train"].select(range(samples_train)),
    eval_dataset = processed_dataset["validation"].select(range(samples_validation)),
    data_collator = data_collate_function,
    #compute_metrics= do_evaluation,
)
trainer

[INFO] Generating trainer with 54936 training items and 8005 validation items.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.027774,0.001672
2,0.001687,0.001642
3,0.001665,0.001625
4,0.001653,0.001613
5,0.001641,0.001605
6,0.001629,0.001600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2580, training_loss=0.006008221936780353, metrics={'train_runtime': 10253.7891, 'train_samples_per_second': 32.146, 'train_steps_per_second': 0.252, 'total_flos': 7.991624898942441e+17, 'train_loss': 0.006008221936780353, 'epoch': 6.0})

In [ ]:
model = trainer.model

In [ ]:
trainer.args = training_args

### Uplad the trained model...

In [ ]:
from google.colab import userdata
# Push our model to the Hugging Face Hub
model_on_hub_url = trainer.push_to_hub(commit_message="Uploaded trained Rostlab/prot_bert model, trained on 54936 items of the data..",
                                                token=userdata.get('Huggingface')
                                                )
print(f"[INFO] Model uploaded to HuggingFace hub with {model_on_hub_url}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...bert_v1/model.safetensors:   2%|2         | 37.6MB / 1.79GB            

  ...bert_v1/training_args.bin:  22%|##1       | 1.14kB / 5.20kB            

[INFO] Model uploaded to HuggingFace hub with https://huggingface.co/Kelmoir/2026-02-14_CAFA6_classification_Rostlab_prot_bert_v1/commit/8c283e31deb527308c5f93579dac5144b07d7e50


### Load the trained model

In [ ]:
#load the model
from google.colab import userdata
# Generate required items from the model
from transformers import AutoModelForSequenceClassification, BertTokenizerFast
def load_model():
  return AutoModelForSequenceClassification.from_pretrained("Kelmoir/2026-02-14_CAFA6_classification_Rostlab_prot_bert_v1",
                                    token=userdata.get("Huggingface"))
def create_tokenizer():
  return BertTokenizerFast.from_pretrained("Rostlab/prot_bert",
                                       do_lower_case=False,
                                       token=userdata.get("Huggingface"))

model = load_model()
tokenizer = create_tokenizer()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.79G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# We can get the evaluate model outputs, we can use the evaluate method of the Trainer

# Re-create the trainer with the newly loaded model to ensure correct accelerate state
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = processed_dataset["train"], #.select(range(2000)),
    eval_dataset = processed_dataset["validation"], #.select(range(200)),
    data_collator = data_collate_function,
    compute_metrics= do_evaluation,
)

test_metric_outputs = trainer.evaluate(processed_dataset["test"].select(range(100)))
test_metric_outputs

{'eval_loss': 0.0020266480278223753,
 'eval_model_preparation_time': 0.0071,
 'eval_accuracy': 0.999732057416268,
 'eval_precision': 0.999732057416268,
 'eval_recall': 0.999732057416268,
 'eval_runtime': 30.4252,
 'eval_samples_per_second': 3.287,
 'eval_steps_per_second': 0.822}

In [ ]:
# Predict and evaluate with the model on cpu

import torch

eval_samples = 50
collated_data_batch = data_collate_function(processed_dataset["test"].select(range(eval_samples)))
model.cpu()
model.eval()
with torch.inference_mode():
  outputs = model(**collated_data_batch)
  for threshold in range(0, 15):
    print(f"The following evaluation uses a threshold of: {threshold/40}")
    print(do_evaluation((outputs["logits"].detach().numpy(), collated_data_batch["labels"].detach().numpy()),
                        threshold = threshold/40))

The following evaluation uses a threshold of: 0.0
{'active_loss': 0.9997397129186603}
The following evaluation uses a threshold of: 0.025
{'active_loss': 0.928798093268445}
The following evaluation uses a threshold of: 0.05
{'active_loss': 0.9044333351833351}
The following evaluation uses a threshold of: 0.075
{'active_loss': 0.8960790119237488}
The following evaluation uses a threshold of: 0.1
{'active_loss': 0.8940415131905979}
The following evaluation uses a threshold of: 0.125
{'active_loss': 0.881040541531467}
The following evaluation uses a threshold of: 0.15
{'active_loss': 0.8933142325510747}
The following evaluation uses a threshold of: 0.175
{'active_loss': 0.8791077441077441}
The following evaluation uses a threshold of: 0.2
{'active_loss': 0.8891077441077441}
The following evaluation uses a threshold of: 0.225
{'active_loss': 0.8891077441077441}
The following evaluation uses a threshold of: 0.25
{'active_loss': 0.8891077441077441}
The following evaluation uses a threshold o

### Evaluate model on test set and active loss

current best with threshold of 0.2:
The final active loss over the test data: 0.8914763662594347

In [ ]:
# Predict with the model
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm as progressbar

# Re-define BATCH_SIZE and DATA_LOADER_NUM_WORKERS as they were not in scope
BATCH_SIZE =  32 #This is HW dependend, you will usually increae this to max outr the HW memmory
DATA_LOADER_NUM_WORKERS = 2

# Create a DataLoader for the test dataset
test_dataloader = DataLoader(
    processed_dataset["test"],
    batch_size=BATCH_SIZE, # Use the batch size defined for training or adjust for inference
    shuffle=False, # No need to shuffle for prediction
    collate_fn=data_collate_function,
    num_workers=DATA_LOADER_NUM_WORKERS # Use the same number of workers as for training
)
# Move the collated batch to the appropriate device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sum_loss = 0
model.to(device) # Ensure the model is also on the correct device
model.eval()
for batch_idx, batch in progressbar(enumerate(test_dataloader), total=len(test_dataloader), desc="Predicting on test"):
  with torch.inference_mode():
    # Move batch items to the appropriate device
    batch_on_device = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

    outputs = model(**batch_on_device)
    # Evaluate each batch
    active_loss = do_evaluation((outputs["logits"].detach().cpu().numpy(), batch["labels"].detach().numpy()), threshold=0.2)["active_loss"]
    #print(f"Active loss in batch {batch_idx}: {active_loss}")
    sum_loss += active_loss
print(f" the final active loss over the test data: {sum_loss/len(test_dataloader)}")

Predicting on test:   0%|          | 0/486 [00:00<?, ?it/s]

 the final active loss over the test data: 0.8914763662594347


In [ ]:

# The protein idds will need to be retrived from the processed dataset
processed_outputs_df = process_results(proteins=processed_dataset["test"][:eval_samples]["protein"],
                                       logits=outputs["logits"],
                                       threshold=0.6)
processed_outputs_df.head(20)

,0,1,2
0,O18400,GO:0005515,0.344
1,O18400,GO:0005737,0.134
2,O18400,GO:0005829,0.133
3,O18400,GO:0005634,0.130
4,O18400,GO:0005886,0.126
5,O18400,GO:0005739,0.081
6,O18400,GO:0005654,0.052
7,O18400,GO:0005576,0.035
8,O18400,GO:0042802,0.035
9,O18400,GO:0016020,0.034


In [ ]:
train_df[train_df['EntryID']=="O08849"]

,EntryID,term,aspect
249948,O08849,GO:0060087,P
249949,O08849,GO:0050873,P
249950,O08849,GO:0005096,F
249951,O08849,GO:0005634,C
249952,O08849,GO:0007186,P
249953,O08849,GO:0010855,F
249954,O08849,GO:0005515,F
249955,O08849,GO:0009898,C
249956,O08849,GO:0007283,P
249957,O08849,GO:0005829,C


## Predict on the final test dataset

We need to generate that first, though.

In [ ]:
import datasets

# Load the test superset and turn them into a dataset
# use a function in order to not waste any memory.
def generate_test_dataset():
  test_prot = []
  test_sequences = []
  for sequence in SeqIO.parse(dataset_dir/"Test/testsuperset.fasta", "fasta"):
    parts = sequence.id.split("|")
    if len(parts) >= 2:
      protein_id = parts[1]
    else:
      protein_id = parts[0]
    test_prot.append(protein_id)
    test_sequences.append(str(sequence.seq).upper().replace(r"[UZOB]", "X"))
  data_dict = {"protein": test_prot, "sequence": test_sequences}
  return (datasets.Dataset.from_dict(data_dict), test_prot)

test_dataset, test_proteins = generate_test_dataset()

test_dataset = test_dataset.with_transform(transform=preprocess_batch_partial)


## Create Test DataLoader

### Subtask:
Create a `torch.utils.data.DataLoader` for the processed `test_dataset`. This DataLoader will use the `data_collate_function` to create batches of data, which is essential for efficient batched predictions on the GPU.


In [ ]:
from torch.utils.data import DataLoader

# Re-define BATCH_SIZE and DATA_LOADER_NUM_WORKERS as they were not in scope
BATCH_SIZE =  128 #This is HW dependend, you will usually increae this to max outr the HW memmory
DATA_LOADER_NUM_WORKERS = 2

# Create a DataLoader for the test dataset
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE, # Use the batch size defined for training or adjust for inference
    shuffle=False, # No need to shuffle for prediction
    collate_fn=data_collate_function,
    num_workers=DATA_LOADER_NUM_WORKERS # Use the same number of workers as for training
)

print(f"Created test DataLoader with batch size {BATCH_SIZE} and {len(test_dataloader)} batches.")

Created test DataLoader with batch size 128 and 1753 batches.


**Reasoning**:
The `test_dataloader` has been successfully created. Now, I will iterate through this `test_dataloader` to generate predictions for the entire test dataset using the loaded model.



In [ ]:
from tqdm.auto import tqdm as progressbar
all_protein_ids = []
all_logits = []

# Ensure model is in evaluation mode and on the correct device
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for batch_idx, batch in progressbar(enumerate(test_dataloader), total=len(test_dataloader), desc="Predicting on test"):
  with torch.inference_mode():
    # Move batch items to the appropriate device
    batch_on_device = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

    outputs = model(**batch_on_device)

    # Detach logits from GPU and convert to numpy
    batch_logits = outputs["logits"].detach().cpu()#.numpy()
    #all_logits.append(batch_logits)

    # Extract protein IDs for the current batch
    # The 'protein' key is directly available from the processed_dataset (before collate_fn)
    # and needs to be manually aligned with the batch outputs.
    # Since test_dataloader is created from test_dataset, we can get protein IDs from test_dataset
    # corresponding to the current batch. The `test_dataset` is `with_transform` so it doesn't have
    # the 'protein' key in the batch after `data_collate_function`. Need to adjust logic.
    # For now, let's assume `test_dataset` has a 'protein' column accessible by index.
    # A more robust way would be to include protein IDs in the collated batch.

    # Retrieve original protein IDs for the current batch from the test_dataset
    batch_protein_ids = test_proteins[batch_idx * BATCH_SIZE: min((batch_idx + 1) * BATCH_SIZE, len(test_dataset))]
    processed_outputs_df = process_results(proteins = batch_protein_ids, logits=batch_logits, threshold = 0.25)
    processed_outputs_df.to_csv(model_data_path/"test_results.tsv", sep="\t", mode="a", header=False,index=False)
    #all_protein_ids.extend(batch_protein_ids)


# Concatenate all logits into a single numpy array
final_logits = np.concatenate(all_logits, axis=0)

print(f"Generated predictions for {len(final_logits)} samples.")
print(f"Shape of final_logits: {final_logits.shape}")
print(f"Number of protein IDs collected: {len(all_protein_ids)}")

Predicting on test:   0%|          | 0/1753 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d0296ba5c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d0296ba5c60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

KeyboardInterrupt: 

In [ ]:
processed_outputs_df = process_results(proteins = all_protein_ids, logits=final_logits, threshold = 0.25)
processed_outputs_df.write_csv(model_data_path/"test_results.tsv", delimiter="\t")

## Huggingface demo application

Now, generate an Huggingface demo application

In [ ]:
import gradio as gr
gr.__version__

We will upload everything programmatically to Hugging Face.

We will need:
- The Path to store the files
  - app.py
  - readme.md
  - requirements.txt
    - examples?

In [ ]:
from pathlib import Path
# save path for the files
save_path = Path("../demos/cafa6_protbert_demo")

# Create the directory
save_path.mkdir(exist_ok=True, parents=True)

### The app.py file

The demo will go from a protein sequence to the predicted OBO-terms. Thus, no need for Biopython, etc.

The expected control flow will be:
- Read in the string
- Prepare for tokenization and then tokenize it.
- Load the model
- Predict with the model
- Post process the outputs - id2label, etc required! part of the model - if not, than the odel has issues!
- Output the results

In [ ]:
%%writefile ../demos/cafa6_protbert_demo/app.py
import gradio as gr
import torch
import pandas as pd
from transformers import AutoModelForSequenceClassification, BertTokenizerFast

#Setting up the model
model_save_path = "Kelmoir/2026-02-14_CAFA6_classification_Rostlab_prot_bert_v1"
model = AutoModelForSequenceClassification.from_pretrained(model_save_path)
tokenizer = BertTokenizerFast.from_pretrained("Rostlab/prot_bert",
                                              do_lower_case=False)

# Setting up the computation device - cuda or CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Get the id2label
id2label = model.config.id2label

model.to(device)

def prepare_for_tokenization(input_prot_seq: str)-> str:
  """
  This function shall insert a [SPACE] between each protein sequence item, and returns the resulting string,
  also, uppercases the sting and replaces unlike proteins
  A maximum length of 1024 is enforced as was used for model training
  """
  if len(input_prot_seq) > 1024:
    input_prot_seq = input_prot_seq[:1024]
  input_prot_seq = input_prot_seq.strip().upper().replace(r"[UZOB]", "X")
  spaces = " "*len(input_prot_seq)

# Source - https://stackoverflow.com/a
# Posted by Ma0, modified by community. See post 'Timeline' for change history
# Retrieved 2026-01-12, License - CC BY-SA 3.0
  return ''.join(map(''.join, zip(input_prot_seq, spaces))).strip()  # kudos @Coldspeed

def process_result(logits, threshold=0.6, top_n=10):
  """
  This function will take in a single protein logit prediction and turns it into alist of predicted GO-terms
  Args:
  **logits** a single Logit array
  **threshold** A float value between 0 and 1. Only GO-terms with a score higher than this value will be considered.
  **top_n** An integer value. If no GO-terms are above the threshold, the top_n GO-terms will be returned.
  Ensure, that the length is equal
  """
  all_results = []
  # Convert logits to probabilities using sigmoid
  probabilities = torch.sigmoid(logits).squeeze()

  # Get indices of GO terms above threshold
  above_threshold_indices = torch.where(probabilities > threshold)[0]

  # If no terms above threshold, take the top_n terms
  if len(above_threshold_indices) == 0:
    top_n_values, top_n_indices = torch.topk(probabilities, k=min(top_n, len(probabilities)))
    selected_indices = top_n_indices
    selected_probabilities = top_n_values
  else:
    selected_indices = above_threshold_indices
    selected_probabilities = probabilities[above_threshold_indices]

  # Sort selected terms by probability in descending order
  sorted_probabilities, sort_indices = torch.sort(selected_probabilities, descending=True)
  sorted_indices = selected_indices[sort_indices]

  for idx, prob in zip(sorted_indices, sorted_probabilities):
    go_term = id2label[idx.item()]
    score = prob.item()
    all_results.append([go_term, f"{score:.3f}"]) # Format score to 3 decimal places

  results_df = pd.DataFrame(all_results)
  # The request specifies 3 columns, all string, no headers, no index
  return results_df

def predict_on_input(input: str, threshold:float):
  model.eval()
  with torch.inference_mode():
    tokenized = tokenizer(prepare_for_tokenization(input),
                          return_tensors="pt").to(device)
    output_logits = model(**tokenized)

  output_table = process_result(output_logits.logits,
                                threshold=threshold)

  return output_table

description = """
This is a demo project to showcase my attempts at predicting the protein functions - known as GO-terms based of the protein amino acid string.

This is essentially the scope of the CAFA 6 challenge that was hosted on Kaggle - https://www.kaggle.com/competitions/cafa-6-protein-function-prediction/overview

This demo takes in a single protein sequence, and will then predict the corresonding GO-terms based of that, and display the score.

Right now, no further post-processing or frills are available.

The model used to perform the predictions is a fine tuned ProtBERT Model. See: https://huggingface.co/Rostlab/prot_bert

And finally, the model isn't to good right now, there is much more experimentation ahead. For instance, the example was reported with the following GO-terms:
- GO:1990837
- GO:0005515
"""

demo = gr.Interface(
    fn = predict_on_input,
    inputs = [
        gr.Textbox(value="MHHRMNEMNLSPVGMEQLTSSSVSNALPVSGSHLGLAASPTHSAIPAPGLPVAIPNLGPSLSSLPSALSLMLPMGIGDRGVMCGLPERNYTLPPPPYPHLESSYFRTILPGILSYLADRPPPQYIHPNSINVDGNTALSITNNPSALDPYQSNGNVGLEPGIVSIDSRSVNTHGAQSLHPSDGHEVALDTAITMENVSRVTSPISTDGMAEELTMDGVAGEHSQIPNGSRSHEPLSVDSVSNNLAADAVGHGGVIPMHGNGLELPVVMETDHIASRVNGMSDSALSDSIHTVAMSTNSVSVALSTSHNLASLESVSLHEVGLSLEPVAVSSITQEVAMGTGHVDVSSDSLSFVSPSLQMEDSNSNKENMATLFTIWCTLCDRAYPSDCPEHGPVTFVPDTPIESRARLSLPKQLVLRQSIVGAEVGVWTGETIPVRTCFGPLIGQQSHSMEVAEWTDKAVNHIWKIYHNGVLEFCIITTDENECNWMMFVRKARNREEQNLVAYPHDGKIFFCTSQDIPPENELLFYYSRDYAQQIGVPEHPDVHLCNCGKECNSYTEFKAHLTSHIHNHLPTQGHSGSHGPSHSKERKWKCSMCPQAFISPSKLHVHFMGHMGMKPHKCDFCSKAFSDPSNLRTHLKIHTGQKNYRCTLCDKSFTQKAHLESHMVIHTGEKNLKCDYCDKLFMRRQDLKQHVLIHTQERQIKCPKCDKLFLRTNHLKKHLNSHEGKRDYVCEKCTKAYLTKYHLTRHLKTCKGPTSSSSAPEEEEEDDSEEEDLADSVGTEDCRINSAVYSADESLSAHK",
                   label="Protein sequence",
                   show_label=True),
        gr.Slider(minimum = -1,
                  maximum = 1,
                  step= 0.01,
                  value=0.25,
                  label="Prediction threshold",
                  show_label=True)
    ],
    outputs = [
        gr.Dataframe(label="Outpt table of Go-terms",
                     show_label=True)
    ],
    description = description,
    title ="CAFA ProtBERT prediction demo V1",
)
#demo.launch(debug=True)
demo.launch()


Writing ../demos/cafa6_protbert_demo/app.py


In [ ]:
%%writefile ../demos/cafa6_protbert_demo/requirements.txt
timm
gradio
torch
transformers

Writing ../demos/cafa6_protbert_demo/requirements.txt


In [ ]:
%%writefile ../demos/cafa6_protbert_demo/README.md
---
title: CAFA ProtBERT demo v1
emoji: 🧬
colorFrom: purple
colorTo: blue
sdk: gradio
sdk_version: 5.50.0
app_file: app.py
pinned: false
license: apache-2.0
---

# CAFA ProtBERT demo v1

This is a demo project to showcase my attempts at predicting the protein functions - known as GO-terms based of the protein amino acid string.

This is essentially the scope of the CAFA 6 challenge that was hosted on Kaggle - https://www.kaggle.com/competitions/cafa-6-protein-function-prediction/overview

This demo takes in a single protein sequence, and will then predict the corresonding GO-terms based of that, and display the score.

Right now, no further post-processing or frills are available.

The model used to perform the predictions is a fine tuned ProtBERT Model. See: https://huggingface.co/Rostlab/prot_bert

And finally, the model isn't to good right now, there is much more experimentation ahead. For instance, the example was reported with the following GO-terms:
- GO:1990837
- GO:0005515

Writing ../demos/cafa6_protbert_demo/README.md


In [ ]:
gr.__version__

NameError: name 'gr' is not defined

### Upload the demo to Hugging face

As per the lessen, we are using the Hugging Face API to upload the demo.

In [ ]:
from huggingface_hub import create_repo, upload_folder, get_full_repo_name, upload_file
from google.colab import userdata

# Set the constants for naming and such
LOCAL_FOLDER_TO_UPLOAD = save_path
HF_REPO_NAME = "CAFA_ProtBERT_Demo_v1"
HF_REPO_TYPE = "space"
HF_SPACE_SDK = "gradio"

# Create the space
print(f"[INFO] Creating the space witht he name of {HF_REPO_NAME}")
create_repo(
    repo_id=HF_REPO_NAME,
    repo_type=HF_REPO_TYPE,
    space_sdk = HF_SPACE_SDK,
    token=userdata.get("Huggingface"),
    private=False,
    exist_ok=True
)

#Get the full reo name
full_hf_repo_name = get_full_repo_name(model_id=HF_REPO_NAME,
                                    token = userdata.get("Huggingface"))
print(f"[INFO] The full HF repo name is: {full_hf_repo_name}")

# Uploading the folder to HF
print(f"[INFO] Uploading the demo folder.")
upload_folder(repo_id = full_hf_repo_name,
              folder_path = LOCAL_FOLDER_TO_UPLOAD,
              token=userdata.get("Huggingface"),
              path_in_repo=".", # Upload to root directory of the repo
              repo_type=HF_REPO_TYPE,
              commit_message= "Various small documentation updates")

[INFO] Creating the space witht he name of CAFA_ProtBERT_Demo_v1
[INFO] The full HF repo name is: Kelmoir/CAFA_ProtBERT_Demo_v1
[INFO] Uploading the demo folder.


CommitInfo(commit_url='https://huggingface.co/spaces/Kelmoir/CAFA_ProtBERT_Demo_v1/commit/022b0d176e0a6a4ab02255b0661632b082837ecc', commit_message='Various small documentation updates', commit_description='', oid='022b0d176e0a6a4ab02255b0661632b082837ecc', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/Kelmoir/CAFA_ProtBERT_Demo_v1', endpoint='https://huggingface.co', repo_type='space', repo_id='Kelmoir/CAFA_ProtBERT_Demo_v1'), pr_revision=None, pr_num=None)

## Ideas for future improvement

- Model:
  - MOE architecture, e.g. First predicting the Classes, then more specialized models to predict the actual GO-terms
  - GO terms don't encompass the entire protein, that should be represented
  - Try with differnet base models or even a fresh network
- Data:
  - Annotate the ranges of the GO terms
  - Add the automatically but reliable annotations to reduce erros in the data
  - Reduce errors in the data - see if bad samples can be estimated.
  - More data
- Post model
  - GO-terms come in graphs, that can potentially be used to weed out small mistakes